<h1 dir="rtl" style="text-align: right;">
تحلیل اکتشافی داده های اضطراب اجتماعی
</h1>

<p dir="rtl" style="text-align: right;">
<strong>فاز دوم: <span dir="ltr">EDA</span></strong>
</p>

<p dir="rtl" style="text-align: right;">
اعضای تیم: علی خوش اخلاق، محمدحسین میرمعصومی، آرمین نورمحمدی، علی کریمی، محسن منصف
</p>

<h2 dir="rtl" style="text-align: right;">
1. کتابخانه ها و تنظیمات
</h2>

<p dir="rtl" style="text-align: right;">
از پایتون 3.13 استفاده کنید و ورژن های زیر
</p>

In [ ]:
# pandas==3.0.5 numpy==2.2.6 plotly==7.1.0 scipy==1.16.3 statsmodels==0.15.0 matplotlib==3.11.2

In [ ]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import scipy
import statsmodels
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pio.renderers.default = "notebook"
px.defaults.template = "plotly_dark"
px.defaults.height = 450
BLUE = "#4C78A8"
RED = "#E45756"
BG = '#F7F7F5'
TARGET = "Anxiety Level (1-10)"
LABEL = "Target"
TEXT = '#22333b'

<h2 dir="rtl" style="text-align: right;">
2. داده های اولیه
</h2>

In [ ]:
df = pd.read_csv("social_anxiety_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Missing_%': (df.isna().mean() * 100).round(1),
    'unique': df.nunique(),
})

In [ ]:
print(df.duplicated().sum())

<h2 dir="rtl" style="text-align: right;">
مشاهدات اولیه
</h2>

<p dir="rtl" style="text-align: right;">
• سطر هامون 2030 تاس و ستون هامون 22 تاس
</p>

<p dir="rtl" style="text-align: right;">
• 9 تا ستون با داده گمشده داریم
</p>

<p dir="rtl" style="text-align: right;">
&nbsp;&nbsp;&nbsp;&nbsp;◦ که <span dir="ltr">Therapy History</span> با 90% داده گمشده بدترینه
</p>

<p dir="rtl" style="text-align: right;">
• <span dir="ltr">Sleep Hours</span>، <span dir="ltr">Physical Activity</span> و <span dir="ltr">Alcohol</span> مقدار منفی دارند
</p>

<p dir="rtl" style="text-align: right;">
• <span dir="ltr">Stress Level</span> مقدار 15 دارد که منطقی نیست
</p>

<p dir="rtl" style="text-align: right;">
• دوتا ستون <span dir="ltr">Target</span> و <span dir="ltr">is_Anxious</span> تغریبا یکسانند
</p>

<p dir="rtl" style="text-align: right;">
• ستون <span dir="ltr">Heart Rate</span> و <span dir="ltr">Caffeine</span> مقدار خارج از بازه دارند
</p>

<h2 dir="rtl" style="text-align: right;">
3. تحلیل و پاک سازی داده
</h2>

<h3 dir="rtl" style="text-align: right;">
3.1 جمعیت شناختی:
<span dir="ltr">Age, Gender, Occupation</span>
</h3>

<h4 dir="rtl" style="text-align: right;">
انتخاب متغیرهای جمعیت‌شناختی
</h4>

<p dir="rtl" style="text-align: right;">
در این بخش، سه متغیر
<span dir="ltr">Age</span>،
<span dir="ltr">Gender</span>
و
<span dir="ltr">Occupation</span>
به‌عنوان متغیرهای جمعیت‌شناختی انتخاب شدند.
برای بررسی و پاک‌سازی این متغیرها، یک کپی مستقل از این سه ستون ساخته می‌شود تا تغییرات این بخش به‌صورت کنترل‌شده انجام شوند.
</p>

In [ ]:
demographic_df = df[
    [
        "Age",
        "Gender",
        "Occupation"
    ]
].copy()

demographic_df.head()

<h4 dir="rtl" style="text-align: right;">
بررسی اولیه متغیرهای جمعیت‌شناختی
</h4>

<p dir="rtl" style="text-align: right;">
پیش از انجام پاک‌سازی، نوع داده، تعداد مقادیر گمشده و تعداد مقادیر یکتای سه متغیر جمعیت‌شناختی بررسی می‌شود.
این مرحله کمک می‌کند مشکلات موجود در داده قبل از هرگونه تغییر شناسایی شوند.
</p>

In [ ]:
pd.DataFrame({
    "Data Type": demographic_df.dtypes,
    "Missing": demographic_df.isna().sum(),
    "Missing %": (demographic_df.isna().mean() * 100).round(1),
    "Unique": demographic_df.nunique()
})

<p dir="rtl" style="text-align: right;">
نتایج بررسی اولیه نشان داد که ستون
<span dir="ltr">Age</span>
دارای ۶۲ مقدار گمشده، معادل حدود ۳.۱ درصد داده‌ها است.
ستون
<span dir="ltr">Gender</span>
نیز دارای ۱۱۹ مقدار گمشده، معادل حدود ۵.۹ درصد داده‌ها است.
</p>

<p dir="rtl" style="text-align: right;">
در ستون
<span dir="ltr">Occupation</span>
هیچ مقدار گمشده‌ای مشاهده نشد.
همچنین این ستون شامل ۱۳ مقدار یکتا است.
بنابراین، در ادامه لازم است ستون‌های
<span dir="ltr">Age</span>
و
<span dir="ltr">Gender</span>
با دقت بیشتری بررسی شوند.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی متغیر سن
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Age</span>
یک متغیر عددی است.
برای بررسی دامنه، مرکز و پراکندگی مقادیر آن، از متد
<span dir="ltr">describe()</span>
استفاده می‌شود.
همچنین دامنه مقادیر برای شناسایی سن‌های نامعتبر یا غیرعادی بررسی می‌شود.
</p>

In [ ]:
demographic_df["Age"].describe()

<p dir="rtl" style="text-align: right;">
آمار توصیفی نشان داد که سن افراد بین ۱۸ تا ۶۴ سال قرار دارد.
میانگین سن حدود ۳۹.۹ سال و میانه برابر با ۴۰ سال است.
بنابراین، در دامنه سن مقادیر نامعتبر واضح مانند سن منفی یا بسیار بزرگ مشاهده نشد.
</p>

<p dir="rtl" style="text-align: right;">
از آنجا که ۶۲ مقدار در ستون
<span dir="ltr">Age</span>
گمشده است، برای حفظ این ردیف‌ها مقادیر گمشده با میانه سن جایگزین می‌شوند.
استفاده از میانه باعث می‌شود مقدار مرکزی داده حفظ شود و نسبت به مقادیر بسیار کوچک یا بزرگ حساسیت کمتری داشته باشد.
</p>

In [ ]:
age_median = demographic_df["Age"].median()

demographic_df["Age"] = (
    demographic_df["Age"]
    .fillna(age_median)
    .astype(int)
)

print("Median Age:", age_median)
print("Missing Age:", demographic_df["Age"].isna().sum())

<h4 dir="rtl" style="text-align: right;">
بررسی متغیر جنسیت
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Gender</span>
یک متغیر دسته‌ای است.
برای بررسی دسته‌های موجود، فراوانی هر دسته و مقادیر گمشده، از متد
<span dir="ltr">value_counts()</span>
استفاده می‌شود.
</p>

In [ ]:
demographic_df["Gender"].value_counts(dropna=False)

<p dir="rtl" style="text-align: right;">
نتایج نشان داد که مقادیر ثبت‌شده در ستون
<span dir="ltr">Gender</span>
فقط شامل سه دسته معتبر
<span dir="ltr">Female</span>،
<span dir="ltr">Male</span>
و
<span dir="ltr">Other</span>
هستند و ناسازگاری در نام‌گذاری دسته‌ها مشاهده نشد.
</p>

<p dir="rtl" style="text-align: right;">
با این حال، ۱۱۹ مقدار گمشده در این ستون وجود دارد.
از آنجا که اطلاعات کافی برای تعیین جنسیت واقعی این افراد وجود ندارد، جایگزینی آن‌ها با پرتکرارترین دسته می‌تواند توزیع داده را به‌صورت مصنوعی تغییر دهد.
بنابراین، این مقادیر با برچسب
<span dir="ltr">Unknown</span>
مشخص می‌شوند.
</p>

In [ ]:
demographic_df["Gender"] = (
    demographic_df["Gender"]
    .fillna("Unknown")
)

demographic_df["Gender"].value_counts(dropna=False)

<h4 dir="rtl" style="text-align: right;">
بررسی متغیر شغل
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Occupation</span>
یک متغیر دسته‌ای است.
برای بررسی دسته‌های موجود، فراوانی هر شغل و وجود مقادیر گمشده، فراوانی مقادیر این ستون بررسی می‌شود.
</p>

In [ ]:
demographic_df["Occupation"].value_counts(dropna=False)

<p dir="rtl" style="text-align: right;">
ستون
<span dir="ltr">Occupation</span>
شامل ۱۳ دسته شغلی است و هیچ مقدار گمشده‌ای در آن وجود ندارد.
همچنین در بررسی دسته‌های موجود، ناسازگاری مشخصی در نام‌گذاری مقادیر مشاهده نشد.
بنابراین، این ستون بدون تغییر باقی می‌ماند.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی نهایی پاک‌سازی داده‌های جمعیت‌شناختی
</h4>

<p dir="rtl" style="text-align: right;">
پس از انجام پاک‌سازی، وضعیت نهایی سه متغیر جمعیت‌شناختی دوباره بررسی می‌شود تا از حذف مقادیر گمشده و صحیح بودن نوع داده‌ها اطمینان حاصل شود.
</p>

In [ ]:
pd.DataFrame({
    "Missing": demographic_df.isna().sum(),
    "Unique": demographic_df.nunique(),
    "Data Type": demographic_df.dtypes
})

<h4 dir="rtl" style="text-align: right;">
جمع‌بندی پاک‌سازی متغیرهای جمعیت‌شناختی
</h4>

<p dir="rtl" style="text-align: right;">
در ستون
<span dir="ltr">Age</span>
تعداد ۶۲ مقدار گمشده با میانه سن، برابر با ۴۰ سال، جایگزین شد و نوع داده این ستون به عدد صحیح تبدیل شد.
</p>

<p dir="rtl" style="text-align: right;">
در ستون
<span dir="ltr">Gender</span>
تعداد ۱۱۹ مقدار گمشده با برچسب
<span dir="ltr">Unknown</span>
مشخص شد تا بدون فرض کردن جنسیت افراد، اطلاعات این ردیف‌ها در دیتاست حفظ شود.
</p>

<p dir="rtl" style="text-align: right;">
ستون
<span dir="ltr">Occupation</span>
فاقد مقدار گمشده یا دسته نامعتبر بود و بدون تغییر باقی ماند.
در پایان، هیچ مقدار گمشده‌ای در سه متغیر جمعیت‌شناختی باقی نماند.
</p>

<h3 dir="rtl" style="text-align: right;">
3.2 سبک زندگی:
<span dir="ltr">Sleep Hours, Physical Activity, Caffeine Intake, Alcohol Consumption, Smoking, Diet Quality</span>
</h3>

In [ ]:
lifestyle_cols = [
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)',
    'Smoking',
    'Diet Quality (1-10)'
]

lifestyle_df = df[lifestyle_cols].copy()
#==================
# Numeric Data
#==================
Sleep_Hours = lifestyle_df['Sleep Hours']
Physical_Activity = lifestyle_df['Physical Activity (hrs/week)']
Caffeine_Intake = lifestyle_df['Caffeine Intake (mg/day)']
Alcohol_Consumption = lifestyle_df['Alcohol Consumption (drinks/week)']

# =========================
# Categorical Data
# =========================
Smoking = lifestyle_df['Smoking']
Diet_Quality = lifestyle_df['Diet Quality (1-10)']

In [ ]:
# =========================
# Plot Numerical Data
# =========================
fig, axes = plt.subplots(4, 1, figsize=(12, 22))

# --- 1. Sleep Hours ---
axes[0].hist(
    Sleep_Hours,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[0].set_title('Distribution of Sleep Hours', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sleep Hours', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)


# --- 2. Physical Activity ---
axes[1].hist(
    Physical_Activity,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[1].set_title('Distribution of Physical Activity', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Physical Activity (hrs/week)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)


# --- 3. Caffeine Intake ---
axes[2].hist(
    Caffeine_Intake,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[2].set_title('Distribution of Caffeine Intake', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Caffeine Intake (mg/day)', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].grid(axis='y', alpha=0.3)


# --- 4. Alcohol Consumption ---
axes[3].hist(
    Alcohol_Consumption,
    bins=10,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[3].set_title('Distribution of Alcohol Consumption', fontsize=14, fontweight='bold')
axes[3].set_xlabel('Alcohol Consumption (drinks/week)', fontsize=11)
axes[3].set_ylabel('Frequency', fontsize=11)
axes[3].grid(axis='y', alpha=0.3)

# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Numeric Variables',
    fontsize=18,
    fontweight='bold',
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# =========================
# Plot categorical Data
# =========================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 1.Smoking ---
smoking_counts = Smoking.value_counts()

axes[0].bar(
    smoking_counts.index.astype(str),
    smoking_counts.values,
    edgecolor='black',
    alpha=0.8
)

axes[0].set_title('Smoking Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Smoking', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)


# --- 2.Diet Quality ---
diet_counts = Diet_Quality.value_counts().sort_index()

axes[1].bar(
    diet_counts.index.astype(str),
    diet_counts.values,
    edgecolor='black',
    alpha=0.8
)

axes[1].set_title('Diet Quality Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diet Quality (1-10)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)


# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Categorical and Discrete Variables',
    fontsize=17,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# remove invlid data from Numeric columns 
# =========================

lifestyle_df.loc[lifestyle_df['Sleep Hours'] < 0, 'Sleep Hours'] = np.nan

lifestyle_df.loc[lifestyle_df['Physical Activity (hrs/week)'] < 0,
       'Physical Activity (hrs/week)'] = np.nan

lifestyle_df.loc[lifestyle_df['Alcohol Consumption (drinks/week)'] < 0,
       'Alcohol Consumption (drinks/week)'] = np.nan

lifestyle_df.loc[lifestyle_df['Caffeine Intake (mg/day)'] == 1500,
                 'Caffeine Intake (mg/day)'] = np.nan

In [ ]:
numeric_cols = [
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)'
]

# Calculate median and replace missing values
for col in numeric_cols:
    median_value = lifestyle_df[col].median()
    lifestyle_df[col] = lifestyle_df[col].fillna(median_value)

In [ ]:
smoking_mode = Smoking.mode()[0]
print('Smoking Mode:', smoking_mode)
lifestyle_df['Smoking'] = lifestyle_df['Smoking'].fillna(smoking_mode)

Diet_Quality_mode = Diet_Quality.mode()[0]
print('Diet Quality:', Diet_Quality_mode)
lifestyle_df['Diet Quality (1-10)'] = lifestyle_df['Diet Quality (1-10)'].fillna(Diet_Quality_mode)

<h3 dir="rtl" style="text-align: right;">
3.3 فیزیولوژیک:
<span dir="ltr">Heart Rate, Breathing Rate, Sweating Level, Dizziness</span>
</h3>

<h4 dir="rtl" style="text-align: right;">
روش بررسی متغیرهای فیزیولوژیک
</h4>

<p dir="rtl" style="text-align: right;">
در این بخش، نوع داده، مقادیر گمشده، دامنه مقادیر و داده‌های پرت چهار متغیر
<span dir="ltr">Heart Rate</span>،
<span dir="ltr">Breathing Rate</span>،
<span dir="ltr">Sweating Level</span>
و
<span dir="ltr">Dizziness</span>
بررسی می‌شوند. برای متغیرهای عددی از روش
<span dir="ltr">IQR</span>
و برای متغیرهای ترتیبی و دسته‌ای از بررسی مقادیر یکتا و فراوانی استفاده می‌شود.
</p>

In [ ]:
# Create a copy and validate physiological variables

physiological_df = df[
    [
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)",
        "Dizziness"
    ]
].copy()

description = physiological_df[
    [
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)"
    ]
].describe()

display(description.T)

# Checking if numerical columns are actually numeric and with no missing values
for col in physiological_df.columns[:3]:
    physiological_df[col] = pd.to_numeric(
        physiological_df[col],
        errors="coerce"
    )

    print(
        col,
        "=>",
        physiological_df[col].isna().sum(),
        "non-numeric or missing values"
    )

print()
print(physiological_df.dtypes)

In [ ]:
# Inspect upper values and detect heart rate outliers using IQR

heart_rate = physiological_df["Heart Rate (bpm)"]

print(
    heart_rate
    .value_counts()
    .sort_index()
    .tail(10)
)

q1 = heart_rate.quantile(0.25)
q3 = heart_rate.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

heart_rate_outliers = physiological_df[
    (heart_rate < lower_bound) |
    (heart_rate > upper_bound)
]

print("\nQ1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(heart_rate_outliers))

print(
    heart_rate_outliers["Heart Rate (bpm)"]
    .value_counts()
    .sort_index()
)

# Inspect related features for heart rate outliers

outlier_indices = heart_rate_outliers.index

outlier_context = df.loc[
    outlier_indices,
    [
        "Age",
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)",
        "Dizziness",
        "Stress Level (1-10)",
        "Anxiety Level (1-10)",
        "Medication"
    ]
]

display(outlier_context.head(10))

# Replace invalid heart rate values with the median

physiological_df.loc[
    physiological_df["Heart Rate (bpm)"] == 220,
    "Heart Rate (bpm)"
] = np.nan

heart_rate_median = physiological_df["Heart Rate (bpm)"].median()

physiological_df["Heart Rate (bpm)"] = (
    physiological_df["Heart Rate (bpm)"]
    .fillna(heart_rate_median)
)

print("Median used:", heart_rate_median)
print(
    "Remaining missing values:",
    physiological_df["Heart Rate (bpm)"].isna().sum()
)

In [ ]:
# Detect breathing rate outliers using IQR

breathing_rate = physiological_df["Breathing Rate (breaths/min)"]

q1_breathing = breathing_rate.quantile(0.25)
q3_breathing = breathing_rate.quantile(0.75)
iqr_breathing = q3_breathing - q1_breathing

lower_bound_breathing = q1_breathing - 1.5 * iqr_breathing
upper_bound_breathing = q3_breathing + 1.5 * iqr_breathing

breathing_outlier_mask = (
    (breathing_rate < lower_bound_breathing) |
    (breathing_rate > upper_bound_breathing)
)

breathing_rate_outliers = physiological_df[breathing_outlier_mask]

print("Q1:", q1_breathing)
print("Q3:", q3_breathing)
print("IQR:", iqr_breathing)
print("Lower bound:", lower_bound_breathing)
print("Upper bound:", upper_bound_breathing)
print("Number of outliers:", breathing_outlier_mask.sum())

print(
    breathing_rate_outliers["Breathing Rate (breaths/min)"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Inspect sweating level values and frequencies

sweating_level = physiological_df["Sweating Level (1-5)"]

print(sweating_level.unique())

print(
    sweating_level
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# Inspect dizziness categories and frequencies

dizziness = physiological_df["Dizziness"]

print(dizziness.unique())
print(dizziness.value_counts(dropna=False))

<h4 dir="rtl" style="text-align: right;">
جمع‌بندی و مشاهدات متغیرهای فیزیولوژیک
</h4>

<p dir="rtl" style="text-align: right;">
• سه متغیر عددی با موفقیت به نوع عددی تبدیل شدند و هیچ مقدار غیرقابل‌تبدیل یا گمشده اولیه در آن‌ها مشاهده نشد.
</p>

<p dir="rtl" style="text-align: right;">
• در متغیر
<strong><span dir="ltr">Heart Rate</span></strong>
تعداد ۳۰ داده پرت شناسایی شد که همگی برابر با
<span dir="ltr">220 bpm</span>
بودند.
</p>

<p dir="rtl" style="text-align: right;">
• تکرار دقیق مقدار ۲۲۰، فاصله زیاد آن با سایر مقادیر و نبود الگوی مشخص در متغیرهای مرتبط، احتمال وجود خطای ثبت یا مقدار نامعتبر سیستماتیک را تقویت می‌کند.
</p>

<p dir="rtl" style="text-align: right;">
• در متغیر
<strong><span dir="ltr">Breathing Rate</span></strong>
هیچ داده پرتی با روش
<span dir="ltr">IQR</span>
شناسایی نشد.
</p>

<p dir="rtl" style="text-align: right;">
• متغیر
<strong><span dir="ltr">Sweating Level</span></strong>
فقط شامل مقادیر معتبر ۱ تا ۵ و متغیر
<strong><span dir="ltr">Dizziness</span></strong>
فقط شامل دسته‌های معتبر
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
بود.
</p>

<h4 dir="rtl" style="text-align: right;">
اقدامات انجام‌شده
</h4>

<p dir="rtl" style="text-align: right;">
• مقادیر نامعتبر ۲۲۰ در ستون
<span dir="ltr">Heart Rate</span>
ابتدا به
<span dir="ltr">NaN</span>
تبدیل و سپس با <strong>میانه مقادیر معتبر</strong> جایگزین شدند.
</p>

<p dir="rtl" style="text-align: right;">
• جایگزینی با میانه باعث حفظ تعداد ردیف‌ها و جلوگیری از ایجاد مقادیر مصنوعی خارج از توزیع اصلی شد؛ با این حال، ممکن است فراوانی داده‌ها را در اطراف میانه کمی افزایش دهد.
</p>

<p dir="rtl" style="text-align: right;">
• متغیرهای
<span dir="ltr">Breathing Rate</span>،
<span dir="ltr">Sweating Level</span>
و
<span dir="ltr">Dizziness</span>
به دلیل نداشتن مقدار نامعتبر، بدون تغییر باقی ماندند.
</p>

<h3 dir="rtl" style="text-align: right;">
3.4 درمان، سابقه و متغیر هدف:<br>

<span dir="ltr">Family History, Medication, Therapy Sessions,</span><br>

<span dir="ltr">Therapy History, Recent Major Life Event, Stress Level, Anxiety Level, Target</span>
</h3>

In [ ]:
treatment_cols = ['Family History of Anxiety', 'Medication', 'Therapy Sessions (per month)', 'Therapy History',
                  'Recent Major Life Event', 'Stress Level (1-10)', 'Anxiety Level (1-10)', 'Target', 'is_Anxious']

treatment_df = df[treatment_cols].copy()

treatment_df.describe(include='all').T

In [ ]:
# آیا Target و is_Anxious یکی هستند؟
same_target_mask = treatment_df['Target'] == treatment_df['is_Anxious']
print(f'Target == is_Anxious در {same_target_mask.sum()} سطر از {treatment_df.shape[0]}')

# Target بررسی
print()
print(treatment_df.groupby('Target')['Anxiety Level (1-10)'].agg(['min', 'max', 'count']))

stress_out_of_scale_mask = treatment_df['Stress Level (1-10)'] > 10

# استرس خارج از مقیاس
print()
print(f'استرس بالای 10: {stress_out_of_scale_mask.sum()} سطر، مقدار: {treatment_df.loc[stress_out_of_scale_mask, "Stress Level (1-10)"].unique()}')


# Therapy History
print()
print(treatment_df['Therapy History'].value_counts(dropna=False))

<h2 dir="rtl" style="text-align: right;">
    مشاهدات
</h2>

<p dir="rtl" style="text-align: right;">
    <strong>is_Anxious</strong> با <strong>Target</strong> کاملاً یکسان است و به نظر می‌رسد
    <strong>is_Anxious</strong> یک کپی از <strong>Target</strong> باشد.
</p>

<h3 dir="rtl" style="text-align: right;">
    تصمیم
</h3>

<p dir="rtl" style="text-align: right;">
    • <strong>Anxiety Level</strong> متغیر اصلی است و <strong>Target</strong> فقط برای رنگ نمودارها و آزمون‌های دسته‌ای استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>is_Anxious</strong> به دلیل تکراری بودن حذف می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>Therapy History</strong> با ۱۸۲۷ مقدار گمشده از ۲۰۳۰ حذف می‌شود؛ با تنها حدود ۱۰٪ داده، نتیجه قابل اتکایی حاصل نمی‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>Stress Level = 15</strong> در ۳۰ سطر → <code>NaN</code> با ثبت فلگ → جایگزینی با <strong>میانه</strong>.
</p>

<p dir="rtl" style="text-align: right;">
    • مقادیر گمشده در <strong>Medication</strong> → <code>Unknown</code>.
</p>

In [ ]:
# فلگ و اصلاح استرس
treatment_df['stress_invalid_flag'] = stress_out_of_scale_mask.astype(int)
treatment_df.loc[stress_out_of_scale_mask, 'Stress Level (1-10)'] = np.nan

median_stress = treatment_df['Stress Level (1-10)'].median()
treatment_df['Stress Level (1-10)'] = treatment_df['Stress Level (1-10)'].fillna(median_stress)

#Medication گمشده حایگزین با Unknown
treatment_df['Medication'] = treatment_df['Medication'].fillna('Unknown')

# حذف ستون های  'is_Anxious', 'Therapy History'
treatment_df = treatment_df.drop(columns=['is_Anxious', 'Therapy History'])

print(f'میانه استرس: {median_stress}')
print(f'ستون های باقی مانده: {treatment_df.shape[1]}')

<h3 dir="rtl" style="text-align: right;">
3.5 ادغام پاک سازی ها و ساخت
<span dir="ltr">clean_df</span>
</h3>

In [ ]:
# Combine cleaned columns
clean_df = pd.concat(
    [
        demographic_df,
        lifestyle_df,
        physiological_df,
        treatment_df
    ],
    axis=1,
    verify_integrity=True
)

clean_df

<h2 dir="rtl" style="text-align: right;">
4. ویژوال تک متغیره
</h2>

<h3 dir="rtl" style="text-align: right;">
4.1 جمعیت شناختی
</h3>

<h4 dir="rtl" style="text-align: right;">
توزیع سن افراد
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Age</span>
یک متغیر عددی است. برای بررسی نحوه توزیع سن افراد از نمودار
<span dir="ltr">Histogram</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
هیستوگرام مقادیر سن را در بازه‌های مختلف گروه‌بندی کرده و تعداد افراد موجود در هر بازه را نمایش می‌دهد.
با استفاده از این نمودار می‌توان شکل کلی توزیع، تمرکز داده‌ها و بازه‌های پرتکرار سن را مشاهده کرد.
</p>

In [ ]:
fig = px.histogram(demographic_df, x="Age",nbins=10, title="Age Distribution")



fig.update_layout(xaxis_title="Age", yaxis_title="Count", bargap=0.3 )

fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی توزیع سن
</h4>

<p dir="rtl" style="text-align: right;">
نمودار توزیع سن نشان می‌دهد که افراد موجود در دیتاست در بازه تقریبی ۱۸ تا ۶۴ سال قرار دارند و داده‌ها در بخش‌های مختلف این بازه پراکنده شده‌اند.
بیشترین فراوانی در محدوده حدود ۴۰ تا ۴۵ سال مشاهده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
با توجه به اینکه در مرحله پاک‌سازی، ۶۲ مقدار گمشده ستون
<span dir="ltr">Age</span>
با میانه سن برابر با ۴۰ جایگزین شدند، بخشی از افزایش فراوانی در اطراف سن ۴۰ می‌تواند ناشی از این جایگزینی باشد.
</p>

<h4 dir="rtl" style="text-align: right;">
توزیع جنسیت
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Gender</span>
یک متغیر دسته‌ای است.
برای مقایسه فراوانی دسته‌های مختلف جنسیت از نمودار
<span dir="ltr">Bar Chart</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
در نمودار میله‌ای، هر میله نمایانگر یکی از دسته‌های متغیر است و ارتفاع آن تعداد افراد موجود در آن دسته را نشان می‌دهد.
</p>

In [ ]:
gender_counts = (
    demographic_df["Gender"]
    .value_counts()
    .reset_index()
)

gender_counts.columns = ["Gender", "Count"]

gender_counts

In [ ]:
fig = px.bar(gender_counts, x="Gender", y="Count", text="Count", title="Gender Distribution")

fig.update_traces(width=0.4)

fig.update_layout(xaxis_title="Gender", yaxis_title="Count" , width=600 , height=400 )

fig.show()

<h4 dir="rtl" style="text-align: right;">
توزیع شغل افراد
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Occupation</span>
یک متغیر دسته‌ای است و شامل ۱۳ دسته شغلی مختلف می‌شود.
برای مقایسه فراوانی شغل‌ها از نمودار
<span dir="ltr">Bar Chart</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
به دلیل تعداد نسبتاً زیاد دسته‌ها، نمودار به‌صورت افقی رسم می‌شود تا نام شغل‌ها خواناتر باشند و مقایسه فراوانی بین دسته‌ها ساده‌تر انجام شود.
</p>

In [ ]:
Occupation_count=(demographic_df["Occupation"].value_counts() .reset_index())
Occupation_count.columns= ['Occupation', 'Count']
Occupation_count

In [ ]:
fig = px.bar(Occupation_count,
    x="Count",
    y="Occupation",
    text="Count",
    orientation="h",
    title="Occupation Distribution")


fig.update_traces(width=0.9)


fig.update_layout( xaxis_title="Count",
    yaxis_title="Occupation")

fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی توزیع شغل
</h4>

<p dir="rtl" style="text-align: right;">
نمودار توزیع شغل نشان می‌دهد که فراوانی دسته‌های مختلف شغلی نسبتاً نزدیک به یکدیگر است و هیچ دسته‌ای سهم غالبی از کل داده‌ها ندارد.
</p>

<p dir="rtl" style="text-align: right;">
در میان دسته‌های موجود،
<span dir="ltr">Student</span>
با ۱۸۲ نفر بیشترین فراوانی را دارد و پس از آن
<span dir="ltr">Artist</span>
با ۱۷۲ نفر و
<span dir="ltr">Lawyer</span>
با ۱۶۷ نفر قرار دارند.
در مقابل،
<span dir="ltr">Freelancer</span>
با ۱۳۴ نفر کمترین فراوانی را دارد.
</p>

<p dir="rtl" style="text-align: right;">
این نمودار تنها توزیع شغل افراد را نمایش می‌دهد و از آن نمی‌توان درباره ارتباط شغل با سطح اضطراب نتیجه‌گیری کرد.
</p>

<h3 dir="rtl" style="text-align: right;">
4.2 سبک زندگی
</h3>

In [ ]:
# =================================
# plot Numeric columns before-after
# =================================
fig, axes = plt.subplots(4, 2, figsize=(14, 22))

cols = [
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)'
]

titles = [
    'Sleep Hours',
    'Physical Activity',
    'Caffeine Intake',
    'Alcohol Consumption'
]

bins_list = [20, 20, 20, 10]

for i, col in enumerate(cols):

# =========================
# 1. df Before
# =========================
    data_before = df[col]
    axes[i, 0].hist(
        data_before,
        bins=bins_list[i],
        color='cornflowerblue',
        edgecolor='black',
        alpha=0.8
    )
    axes[i, 0].set_title(f'{titles[i]} - Before', fontsize=13, fontweight='bold')
    axes[i, 0].set_xlabel(col, fontsize=11)
    axes[i, 0].set_ylabel('Frequency', fontsize=11)
    axes[i, 0].grid(axis='y', alpha=0.3)

# =========================
# 2. df after
# =========================
    data_after = clean_df[col]
    axes[i, 1].hist(
        data_after,
        bins=bins_list[i],
        color='#FF6D00',
        edgecolor='black',
        alpha=0.8
    )
    axes[i, 1].set_title(f'{titles[i]} - After', fontsize=13, fontweight='bold')
    axes[i, 1].set_xlabel(col, fontsize=11)
    axes[i, 1].set_ylabel('Frequency', fontsize=11)
    axes[i, 1].grid(axis='y', alpha=0.3)

# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Numeric Variables: Before vs After Cleaning',
    fontsize=18,
    fontweight='bold',
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# ==================================
# plot Categorical Data before-after
# ==================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# =========================
# 1. Smoking - Before
# =========================
smoking_before = df['Smoking'].value_counts()

axes[0, 0].bar(
    smoking_before.index.astype(str),
    smoking_before.values,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[0, 0].set_title('Smoking - Before', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Smoking', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].grid(axis='y', alpha=0.3)

# =========================
# 2. Smoking - After
# =========================
smoking_after = clean_df['Smoking'].value_counts()

axes[0, 1].bar(
    smoking_after.index.astype(str),
    smoking_after.values,
    color='#FF6D00',
    edgecolor='black',
    alpha=0.8
)
axes[0, 1].set_title('Smoking - After', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Smoking', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].grid(axis='y', alpha=0.3)

# =========================
# 3. Diet Quality - Before
# =========================
diet_before = df['Diet Quality (1-10)'].value_counts().sort_index()

axes[1, 0].bar(
    diet_before.index.astype(str),
    diet_before.values,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[1, 0].set_title('Diet Quality - Before', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Diet Quality (1-10)', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].grid(axis='y', alpha=0.3)

# =========================
# 4. Diet Quality - After
# =========================
diet_after = clean_df['Diet Quality (1-10)'].value_counts().sort_index()

axes[1, 1].bar(
    diet_after.index.astype(str),
    diet_after.values,
    color='#FF6D00',
    edgecolor='black',
    alpha=0.8
)
axes[1, 1].set_title('Diet Quality - After', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Diet Quality (1-10)', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].grid(axis='y', alpha=0.3)

# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Categorical and Discrete Variables: Before vs After',
    fontsize=16,
    fontweight='bold',
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

<h3 dir="rtl" style="text-align: right;">
4.3 فیزیولوژیک
</h3>

<p dir="rtl" style="text-align: right;">
در این بخش، توزیع تک‌متغیره چهار ویژگی فیزیولوژیک شامل ضربان قلب، نرخ تنفس، سطح تعریق و سرگیجه بررسی می‌شود.
برای متغیرهای عددی پیوسته از هیستوگرام به همراه نمودار جعبه‌ای و برای متغیرهای ترتیبی و دسته‌ای از نمودار میله‌ای استفاده شده است.
</p>

<p dir="rtl" style="text-align: right;">
برای بررسی تأثیر مرحله پاک‌سازی بر توزیع ضربان قلب، نمودارهای قبل و بعد از جایگزینی مقادیر نامعتبر نیز با یکدیگر مقایسه می‌شوند.
</p>

In [ ]:
# Compare heart rate distributions before and after cleaning

heart_rate_before_fig = px.histogram(
    df,
    x="Heart Rate (bpm)",
    nbins=30,
    marginal="box",
    title="Distribution of Heart Rate Before Cleaning",
    color_discrete_sequence=[RED]
)

heart_rate_before_fig.update_layout(
    xaxis_title="Heart Rate (bpm)",
    yaxis_title="Number of Participants",
    bargap=0.4,
    showlegend=False,
    title_x=0.5
)

heart_rate_after_fig = px.histogram(
    physiological_df,
    x="Heart Rate (bpm)",
    nbins=30,
    marginal="box",
    title="Distribution of Heart Rate After Cleaning",
    color_discrete_sequence=[BLUE]
)

heart_rate_after_fig.update_layout(
    xaxis_title="Heart Rate (bpm)",
    yaxis_title="Number of Participants",
    bargap=0.4,
    showlegend=False,
    title_x=0.5
)

heart_rate_before_fig.show()
heart_rate_after_fig.show()

In [ ]:
# Visualize the breathing rate distribution using one-unit intervals
breathing_rate_fig = px.histogram(
    physiological_df,
    x="Breathing Rate (breaths/min)",
    nbins=18,
    marginal="box",
    title="Distribution of Breathing Rate",
    color_discrete_sequence=[BLUE]
)

breathing_rate_fig.update_traces(
    xbins=dict(
        start=11.5,
        end=29.5,
        size=1
    ),
    selector=dict(type="histogram")
)

breathing_rate_fig.update_layout(
    xaxis_title="Breathing Rate (breaths/min)",
    yaxis_title="Number of Participants",
    bargap=0.4,
    showlegend=False,
    title_x=0.5
)

breathing_rate_fig.show()

In [ ]:
# Calculate and visualize the frequency of each sweating level
sweating_counts = (
    physiological_df["Sweating Level (1-5)"]
    .value_counts()
    .sort_index()
    .rename_axis("Sweating Level (1-5)")
    .reset_index(name="Count")
)

sweating_fig = px.bar(
    sweating_counts,
    x="Sweating Level (1-5)",
    y="Count",
    text="Count",
    title="Distribution of Sweating Level",
    color_discrete_sequence=[BLUE]
)

sweating_fig.update_traces(
    textposition="outside",
    width=0.35
)

sweating_fig.update_layout(
    xaxis_title="Sweating Level",
    yaxis_title="Number of Participants",
    xaxis=dict(
        tickmode="array",
        tickvals=[1, 2, 3, 4, 5]
    ),
    showlegend=False,
    title_x=0.5
)

sweating_fig.show()

In [ ]:
# Calculate and visualize the frequency of dizziness responses
Dizziness_count = (
    physiological_df["Dizziness"]
    .value_counts()
    .reindex(["No", "Yes"])
    .rename_axis("Dizziness")
    .reset_index(name="Count")
)

Dizziness_fig = px.bar(
    Dizziness_count,
    y="Dizziness",
    x="Count",
    orientation="h",
    text="Count",
    title="Distribution of Dizziness",
    color_discrete_sequence=[BLUE]
)

Dizziness_fig.update_traces(
    textposition="outside",
    cliponaxis=False,
    width=0.4
)

Dizziness_fig.update_layout(
    xaxis_title="Number of Participants",
    yaxis_title="Dizziness",
    showlegend=False,
    title_x=0.5
)

Dizziness_fig.show()

<h4 dir="rtl" style="text-align: right;">
جمع‌بندی مشاهدات فیزیولوژیک
</h4>

<ul dir="rtl" style="text-align: right;">
    <li>
        در داده‌های اولیه ضربان قلب، تعداد ۳۰ مشاهده با مقدار
        <span dir="ltr">220 bpm</span>
        از سایر مقادیر فاصله زیادی داشتند. پس از تبدیل این مقادیر به داده گمشده و جایگزینی آن‌ها با میانه
        <span dir="ltr">92 bpm</span>،
        نقطه پرت جداشده از توزیع حذف شد.
    </li>
    <li>
        افزایش فراوانی در مقدار میانه نمودار ضربان قلب پس از پاک‌سازی، تا حدی نتیجه جایگزینی مقادیر نامعتبر با میانه است و نباید کاملاً به‌عنوان یک الگوی طبیعی تفسیر شود.
    </li>
    <li>
        نرخ تنفس در بازه
        <span dir="ltr">12–29 breaths/min</span>
        قرار دارد و مطابق بررسی انجام‌شده با روش
        <span dir="ltr">IQR</span>،
        داده پرت مشخصی در آن مشاهده نشد.
    </li>
    <li>
        سطح تعریق در هر پنج سطح مشاهده می‌شود. سطح ۴ با ۴۳۶ مشاهده بیشترین و سطح ۱ با ۳۶۰ مشاهده کمترین فراوانی را دارد؛ با این حال عدم تعادل شدیدی میان سطوح دیده نمی‌شود.
    </li>
    <li>
        متغیر سرگیجه تقریباً متعادل است؛ ۱۰۴۹ نفر معادل ۵۱٫۷ درصد پاسخ
        <span dir="ltr">Yes</span>
        و ۹۸۱ نفر معادل ۴۸٫۳ درصد پاسخ
        <span dir="ltr">No</span>
        دارند.
    </li>
</ul>

<h3 dir="rtl" style="text-align: right;">
4.4 درمان، سابقه و متغیر هدف
</h3>

In [ ]:
#داده های خام برای نمودار قبل از پاکسازی
raw_df = pd.read_csv('social_anxiety_dataset.csv')
anxiety_counts = df[TARGET].value_counts().sort_index()
anxiety_counts

In [ ]:
# جدا کردن دو گروه برچسب تا در هیستوگرام با دو رنگ باشند
not_anxious_mask = df[LABEL] == 0
anxious_mask = df[LABEL] == 1

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=('Anxiety Level: most score 2 to 5', 'Boxplot')
)
# غیرمضطرب ها
fig.add_trace(
    go.Histogram(
        x=df.loc[not_anxious_mask, TARGET],
        name='not anxious (Target = 0)',
        marker_color=BLUE,
        xbins={'size': 1}
    ),
    row=1,
    col=1
)
#مضطرب ها 
fig.add_trace(
    go.Histogram(
        x=df.loc[anxious_mask, TARGET],
        name='anxious (Target = 1)',
        marker_color=RED,
        xbins={'size': 1}
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Box(
        y=df[TARGET],
        name=TARGET,
        marker_color=BLUE,
        boxmean=True,
        showlegend=False
    ),
    row=1,
    col=2
)

fig.update_layout(
    title=f'Only {df[LABEL].mean() * 100:.1f}% are anxious (7 and above)',
    barmode='stack'
)
fig.update_xaxes(title_text=TARGET, dtick=1, row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.show()

In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=(
        'Before cleaning: 30 rows at 15, outside the 1 to 10 scale',
        'Boxplot before',
        f'After cleaning: 15 replaced with median ({median_stress})',
        'Boxplot after'
    )
)

fig.add_trace(
    go.Histogram(
        x=raw_df['Stress Level (1-10)'],
        marker_color=RED,
        xbins={'size': 1},
        showlegend=False
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Box(
        y=raw_df['Stress Level (1-10)'],
        marker_color=RED,
        boxmean=True,
        showlegend=False
    ),
    row=1,
    col=2
)
fig.add_trace(
    go.Histogram(
        x=df['Stress Level (1-10)'],
        marker_color=BLUE,
        xbins={'size': 1},
        showlegend=False
    ),
    row=2,
    col=1
)
fig.add_trace(
    go.Box(
        y=df['Stress Level (1-10)'],
        marker_color=BLUE,
        boxmean=True,
        showlegend=False
    ),
    row=2,
    col=2
)

fig.update_layout(title='Stress Level before and after cleaning', height=700)
fig.update_xaxes(title_text='Stress Level (1-10)', dtick=1, row=1, col=1)
fig.update_xaxes(title_text='Stress Level (1-10)', dtick=1, row=2, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.show()

In [ ]:
anxiety_counts = df.groupby([TARGET, LABEL]).size().unstack(fill_value=0)
anxiety_counts

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=('Most people have 1 to 3 sessions per month', 'Boxplot')
)

fig.add_trace(
    go.Histogram(
        x=df['Therapy Sessions (per month)'],
        marker_color=BLUE,
        xbins={'size': 1},
        showlegend=False
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Box(
        y=df['Therapy Sessions (per month)'],
        marker_color=BLUE,
        boxmean=True,
        showlegend=False
    ),
    row=1,
    col=2
)

fig.update_layout(title='Therapy Sessions (per month)')
fig.update_xaxes(title_text='Therapy Sessions (per month)', dtick=1, row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.show()

In [ ]:
medication_before = raw_df['Medication'].fillna('NaN').value_counts()
medication_after = df['Medication'].value_counts()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(f'Before: {raw_df["Medication"].isnull().sum()} missing', 'After: missing kept as Unknown')
)

fig.add_trace(
    go.Bar(
        x=medication_before.index,
        y=medication_before.values,
        marker_color=RED,
        text=medication_before.values,
        showlegend=False
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Bar(
        x=medication_after.index,
        y=medication_after.values,
        marker_color=BLUE,
        text=medication_after.values,
        showlegend=False
    ),
    row=1,
    col=2
)

fig.update_layout(title='Medication before and after cleaning')
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.show()

In [ ]:
binary_cols = ['Family History of Anxiety', 'Medication', 'Recent Major Life Event']
answer_colors = {'Yes': RED, 'No': BLUE, 'Unknown': '#BBBBBB'}

fig = go.Figure()
for answer, color in answer_colors.items():
    counts = [(df[col] == answer).sum() for col in binary_cols]
    fig.add_trace(
        go.Bar(
            x=binary_cols,
            y=counts,
            name=answer,
            marker_color=color,
            text=counts
        )
    )

fig.update_layout(
    title='Family history, medication and life event',
    yaxis_title='Count',
    barmode='group'
)
fig.show()

<h2 dir="rtl" style="text-align: right;">
5. ویژوال دومتغیره
</h2>

<h3 dir="rtl" style="text-align: right;">
5.1 ستون های دسته ای با اضطراب
</h3>

<h4 dir="rtl" style="text-align: right;">
بررسی متغیرهای دسته‌ای در ارتباط با سطح اضطراب
</h4>

<p dir="rtl" style="text-align: right;">
در این بخش، توزیع متغیر
<span dir="ltr">Anxiety Level (1-10)</span>
در گروه‌های مختلف متغیرهای دسته‌ای و ترتیبی موجود در
<span dir="ltr">clean_df</span>
بررسی می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
هدف این تحلیل، مقایسه توزیع سطح اضطراب بین دسته‌های مختلف هر ویژگی است.
برای این منظور از نمودارهایی مانند
<span dir="ltr">Box Plot</span>
استفاده می‌شود که امکان مقایسه میانه، پراکندگی و دامنه مقادیر اضطراب را بین گروه‌ها فراهم می‌کند.
</p>

In [ ]:
# تعریف ستون‌های دسته‌ای و ترتیبی که در تحلیل دومتغیره با سطح اضطراب بررسی می‌شوند
categorical_cols = [
    "Gender",
    "Occupation",
    "Smoking",
    "Diet Quality (1-10)",
    "Sweating Level (1-5)",
    "Dizziness",
    "Family History of Anxiety",
    "Medication",
    "Recent Major Life Event"
]

# نمایش لیست ستون‌های انتخاب‌شده برای اطمینان از صحت متغیرهای مورد بررسی
categorical_cols

<h4 dir="rtl" style="text-align: right;">
بررسی سطح اضطراب بر اساس جنسیت
</h4>

<p dir="rtl" style="text-align: right;">
برای بررسی تفاوت توزیع
<span dir="ltr">Anxiety Level (1-10)</span>
بین گروه‌های مختلف
<span dir="ltr">Gender</span>
از نمودار
<span dir="ltr">Box Plot</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
این نمودار امکان مقایسه میانه، پراکندگی و دامنه سطح اضطراب را بین دسته‌های مختلف جنسیت فراهم می‌کند.
</p>

In [ ]:
# رسم Box Plot برای مقایسه توزیع سطح اضطراب در گروه‌های مختلف جنسیت
fig=px.box(clean_df ,
    x="Gender" , 
     y="Anxiety Level (1-10)" ,
      title= "Anxiety level by Gender" )

fig.update_traces(width=0.2)
# مشخص کردن عنوان محورهای نمودار برای خوانایی بهتر
fig.update_layout( xaxis_title= "Gender" , 
    yaxis_title="Anxiety_Level" , width=800
)

# نمایش نمودار
fig.show()

<h4 dir="rtl" style="text-align: right;">
بررسی سطح اضطراب بر اساس شغل
</h4>

<p dir="rtl" style="text-align: right;">
برای بررسی تفاوت توزیع
<span dir="ltr">Anxiety Level (1-10)</span>
بین گروه‌های مختلف شغلی، از نمودار
<span dir="ltr">Box Plot</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Occupation</span>
یک متغیر دسته‌ای است و شامل چندین گروه شغلی مختلف می‌شود.
نمودار
<span dir="ltr">Box Plot</span>
امکان مقایسه میانه، پراکندگی و دامنه سطح اضطراب را بین این گروه‌های شغلی فراهم می‌کند.
</p>

<p dir="rtl" style="text-align: right;">
این نمودار برای شناسایی تفاوت‌های احتمالی در توزیع سطح اضطراب میان مشاغل مختلف استفاده می‌شود؛ با این حال، مشاهده تفاوت در نمودار به‌تنهایی به معنی وجود تفاوت معنادار آماری نیست و برای بررسی معناداری، آزمون آماری جداگانه مورد نیاز است.
</p>

In [ ]:
# رسم Box Plot برای مقایسه توزیع سطح اضطراب در گروه‌های شغلی
fig = px.box(
    clean_df,
    x="Occupation",
    y="Anxiety Level (1-10)",
    title="Anxiety Level by Occupation"
)

# کاهش عرض Boxها برای خوانایی بهتر نمودار
fig.update_traces(width=0.2)

# تنظیم عنوان محورهای نمودار و عرض کلی شکل
fig.update_layout(
    xaxis_title="Occupation",
    yaxis_title="Anxiety Level",
    width=1100
)

# نمایش نمودار
fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی سطح اضطراب بر اساس شغل
</h4>

<p dir="rtl" style="text-align: right;">
نمودار
<span dir="ltr">Box Plot</span>
نشان می‌دهد که توزیع سطح اضطراب در بیشتر گروه‌های شغلی هم‌پوشانی قابل‌توجهی دارد و میانه سطح اضطراب در اکثر مشاغل در محدوده تقریبی ۳ تا ۴ قرار گرفته است.
</p>

<p dir="rtl" style="text-align: right;">
در برخی گروه‌های شغلی، مقادیر بالاتر اضطراب مانند ۹ و ۱۰ به‌صورت نقاط دور از بخش اصلی توزیع مشاهده می‌شوند.
با این حال، از نظر بصری تفاوت بسیار شدیدی میان توزیع اضطراب در مشاغل مختلف دیده نمی‌شود.
</p>

<p dir="rtl" style="text-align: right;">
این مشاهده تنها یک مقایسه توصیفی است و برای مشخص کردن اینکه تفاوت میان گروه‌های شغلی از نظر آماری معنادار است یا خیر، انجام آزمون آماری مناسب ضروری است.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی سطح اضطراب بر اساس وضعیت سیگار کشیدن
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Smoking</span>
یک متغیر دسته‌ای دودویی است.
برای مقایسه سطح اضطراب بین افراد سیگاری و غیرسیگاری، میانگین
<span dir="ltr">Anxiety Level (1-10)</span>
در هر گروه محاسبه و با استفاده از نمودار
<span dir="ltr">Bar Chart</span>
نمایش داده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
ارتفاع هر میله نشان‌دهنده میانگین سطح اضطراب در یکی از دو گروه
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
است.
</p>

In [ ]:
# محاسبه میانگین سطح اضطراب برای هر گروه از متغیر Smoking
smoking_anxiety = (
    clean_df
    .groupby("Smoking", as_index=False)["Anxiety Level (1-10)"]
    .mean()
)

# رسم نمودار میله‌ای برای مقایسه میانگین اضطراب افراد سیگاری و غیرسیگاری
fig = px.bar(
    smoking_anxiety,
    x="Smoking",
    y="Anxiety Level (1-10)",
    text="Anxiety Level (1-10)",
    title="Mean Anxiety Level by Smoking Status"
)

# گرد کردن اعداد نمایش‌داده‌شده روی میله‌ها
fig.update_traces(
    texttemplate="%{text:.2f}",
    width=0.25
)

# تنظیم عنوان محورهای نمودار
fig.update_layout(
    xaxis_title="Smoking",
    yaxis_title="Mean Anxiety Level" , width =800
)

# نمایش نمودار
fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی سطح اضطراب بر اساس وضعیت سیگار کشیدن
</h4>

<p dir="rtl" style="text-align: right;">
نتایج نشان می‌دهد که میانگین سطح اضطراب در افراد غیرسیگاری برابر با حدود
<span dir="ltr">3.71</span>
و در افراد سیگاری برابر با حدود
<span dir="ltr">4.10</span>
است.
</p>

<p dir="rtl" style="text-align: right;">
بنابراین، در این دیتاست میانگین سطح اضطراب افراد سیگاری حدود
<span dir="ltr">0.39</span>
واحد بیشتر از افراد غیرسیگاری مشاهده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
این اختلاف یک مشاهده توصیفی است و به‌تنهایی نشان‌دهنده رابطه علّی یا تفاوت معنادار آماری نیست.
برای بررسی معناداری این اختلاف، استفاده از آزمون آماری مناسب ضروری است.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی سطح اضطراب بر اساس وجود سرگیجه
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Dizziness</span>
یک متغیر دسته‌ای دودویی شامل دو گروه
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
است.
</p>

<p dir="rtl" style="text-align: right;">
برای مقایسه سطح اضطراب بین افراد دارای سرگیجه و افراد بدون سرگیجه، ابتدا میانگین
<span dir="ltr">Anxiety Level (1-10)</span>
در هر گروه محاسبه شده و سپس با استفاده از نمودار
<span dir="ltr">Bar Chart</span>
نمایش داده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
ارتفاع هر میله نشان‌دهنده میانگین سطح اضطراب در هر گروه است و امکان مقایسه مستقیم میانگین‌ها را فراهم می‌کند.
</p>

In [ ]:
# گروه‌بندی داده‌ها بر اساس وجود یا عدم وجود سرگیجه
# و محاسبه میانگین سطح اضطراب برای هر گروه
dizziness_anxiety = (
    clean_df
    .groupby("Dizziness", as_index=False)["Anxiety Level (1-10)"]
    .mean()
)

# رسم نمودار میله‌ای برای مقایسه میانگین اضطراب
# بین افراد دارای سرگیجه و افراد بدون سرگیجه
fig = px.bar(
    dizziness_anxiety,
    x="Dizziness",
    y="Anxiety Level (1-10)",
    title="Mean Anxiety Level by Dizziness",
    text="Anxiety Level (1-10)"
)

# تنظیم عرض میله‌ها
fig.update_traces(texttemplate="%{text:.2f}"
     ,width=0.35)

# تنظیم عنوان محورهای نمودار
fig.update_layout(
    xaxis_title="Dizziness",
    yaxis_title="Mean Anxiety Level" ,  width=800 , height=550

)

# نمایش نمودار
fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی سطح اضطراب بر اساس وجود سرگیجه
</h4>

<p dir="rtl" style="text-align: right;">
نتایج نشان می‌دهد که میانگین سطح اضطراب در افراد بدون سرگیجه حدود
<span dir="ltr">3.75</span>
و در افراد دارای سرگیجه حدود
<span dir="ltr">4.09</span>
است.
</p>

<p dir="rtl" style="text-align: right;">
بنابراین، در این دیتاست میانگین سطح اضطراب افراد دارای سرگیجه حدود
<span dir="ltr">0.34</span>
واحد بیشتر از افراد بدون سرگیجه مشاهده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
این تفاوت یک مشاهده توصیفی است و به‌تنهایی نشان‌دهنده رابطه علّی یا تفاوت معنادار آماری نیست.
برای بررسی معناداری این اختلاف، انجام آزمون آماری مناسب ضروری است.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی سطح اضطراب بر اساس سابقه خانوادگی اضطراب
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Family History of Anxiety</span>
یک متغیر دسته‌ای دودویی شامل دو گروه
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
است.
</p>

<p dir="rtl" style="text-align: right;">
برای مقایسه سطح اضطراب بین افراد دارای سابقه خانوادگی اضطراب و افراد بدون سابقه خانوادگی، ابتدا میانگین
<span dir="ltr">Anxiety Level (1-10)</span>
برای هر گروه محاسبه شده و سپس با استفاده از نمودار
<span dir="ltr">Bar Chart</span>
نمایش داده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
ارتفاع هر میله نشان‌دهنده میانگین سطح اضطراب در هر گروه است و امکان مقایسه مستقیم این دو گروه را فراهم می‌کند.
</p>

In [ ]:
# گروه‌بندی داده‌ها بر اساس وجود یا عدم وجود سابقه خانوادگی اضطراب
# و محاسبه میانگین سطح اضطراب برای هر گروه
family_history_anxiety = (
    clean_df
    .groupby("Family History of Anxiety", as_index=False)["Anxiety Level (1-10)"]
    .mean()
)

# نمایش جدول خلاصه برای بررسی میانگین‌ها
family_history_anxiety
# رسم نمودار میله‌ای برای مقایسه میانگین اضطراب
# بین افراد دارای سابقه خانوادگی اضطراب و افراد بدون سابقه
fig = px.bar(
    family_history_anxiety,
    x="Family History of Anxiety",
    y="Anxiety Level (1-10)",
    text="Anxiety Level (1-10)",
    title="Mean Anxiety Level by Family History of Anxiety"
)

# نمایش میانگین‌ها با دو رقم اعشار و تنظیم عرض میله‌ها
fig.update_traces(
    texttemplate="%{text:.2f}",
    width=0.35
)

# تنظیم عنوان محورهای نمودار
fig.update_layout(
    xaxis_title="Family History of Anxiety",
    yaxis_title="Mean Anxiety Level",
    width=800,
    height=550
)

# نمایش نمودار
fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی سطح اضطراب بر اساس سابقه خانوادگی اضطراب
</h4>

<p dir="rtl" style="text-align: right;">
نتایج نشان می‌دهد که میانگین سطح اضطراب در افراد بدون سابقه خانوادگی اضطراب حدود
<span dir="ltr">3.46</span>
و در افراد دارای سابقه خانوادگی اضطراب حدود
<span dir="ltr">4.35</span>
است.
</p>

<p dir="rtl" style="text-align: right;">
بنابراین، در این دیتاست میانگین سطح اضطراب افراد دارای سابقه خانوادگی اضطراب حدود
<span dir="ltr">0.89</span>
واحد بیشتر از افرادی است که سابقه خانوادگی اضطراب ندارند.
</p>

<p dir="rtl" style="text-align: right;">
این تفاوت یک مشاهده توصیفی است و به‌تنهایی برای نتیجه‌گیری درباره رابطه علّی یا معناداری آماری کافی نیست.
برای بررسی معناداری این اختلاف، انجام آزمون آماری مناسب ضروری است.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی سطح اضطراب بر اساس مصرف دارو
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Medication</span>
یک متغیر دسته‌ای است که وضعیت مصرف دارو را در گروه‌های مختلف نمایش می‌دهد.
</p>

<p dir="rtl" style="text-align: right;">
برای مقایسه توزیع
<span dir="ltr">Anxiety Level (1-10)</span>
بین گروه‌های مختلف این متغیر، از نمودار
<span dir="ltr">Box Plot</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
این نمودار امکان مقایسه میانه، پراکندگی و دامنه سطح اضطراب را در افراد دارای وضعیت‌های مختلف مصرف دارو فراهم می‌کند.
</p>

In [ ]:
# رسم Box Plot برای مقایسه توزیع سطح اضطراب
# در گروه‌های مختلف وضعیت مصرف دارو
fig = px.box(
    clean_df,
    x="Medication",
    y="Anxiety Level (1-10)",
    title="Anxiety Level by Medication"
)

# کاهش عرض Boxها برای خوانایی بهتر نمودار
fig.update_traces(
    width=0.25
)

# تنظیم عنوان محورهای نمودار و ابعاد شکل
fig.update_layout(
    xaxis_title="Medication",
    yaxis_title="Anxiety Level",
    width=650,
    height=450
)

# نمایش نمودار
fig.show()

<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی سطح اضطراب بر اساس مصرف دارو
</h4>

<p dir="rtl" style="text-align: right;">
نمودار
<span dir="ltr">Box Plot</span>
نشان می‌دهد که میانه سطح اضطراب در گروه
<span dir="ltr">Yes</span>
حدود ۴ است، در حالی که میانه در گروه‌های
<span dir="ltr">No</span>
و
<span dir="ltr">Unknown</span>
حدود ۳ مشاهده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
با وجود این تفاوت، دامنه و بخش قابل‌توجهی از توزیع سطح اضطراب میان گروه‌ها هم‌پوشانی دارد.
همچنین مقادیر بالای اضطراب در هر سه گروه مشاهده می‌شوند.
</p>

<p dir="rtl" style="text-align: right;">
بنابراین، از نظر توصیفی گروه مصرف‌کننده دارو سطح اضطراب بالاتری نشان می‌دهد، اما این نمودار به‌تنهایی برای نتیجه‌گیری درباره رابطه علّی یا معناداری آماری کافی نیست.
</p>

<h4 dir="rtl" style="text-align: right;">
بررسی سطح اضطراب بر اساس تجربه رویداد مهم اخیر در زندگی
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Recent Major Life Event</span>
یک متغیر دسته‌ای دودویی شامل دو گروه
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
است.
</p>

<p dir="rtl" style="text-align: right;">
برای مقایسه سطح اضطراب میان افرادی که اخیراً یک رویداد مهم در زندگی تجربه کرده‌اند و افرادی که چنین تجربه‌ای نداشته‌اند، میانگین
<span dir="ltr">Anxiety Level (1-10)</span>
برای هر گروه محاسبه و با نمودار
<span dir="ltr">Bar Chart</span>
نمایش داده می‌شود.
</p>

In [ ]:
# گروه‌بندی داده‌ها بر اساس تجربه یا عدم تجربه رویداد مهم اخیر
# و محاسبه میانگین سطح اضطراب در هر گروه
life_event_anxiety = ( clean_df.groupby("Recent Major Life Event", as_index=False)["Anxiety Level (1-10)"].mean()
)
# رسم نمودار میله‌ای برای مقایسه میانگین سطح اضطراب
# بین افراد دارای و فاقد رویداد مهم اخیر در زندگی
fig = px.bar(
    life_event_anxiety,
    x="Recent Major Life Event",
    y="Anxiety Level (1-10)",
    text="Anxiety Level (1-10)",
    title="Mean Anxiety Level by Recent Major Life Event"
)

# نمایش مقدار میانگین با دو رقم اعشار و تنظیم عرض میله‌ها
fig.update_traces(
    texttemplate="%{text:.2f}",
    width=0.35
)

# تنظیم عنوان محورهای نمودار و ابعاد شکل
fig.update_layout(
    xaxis_title="Recent Major Life Event",
    yaxis_title="Mean Anxiety Level",
    width=800,
    height=550
)

# نمایش نمودار
fig.show()


<h4 dir="rtl" style="text-align: right;">
نتیجه بررسی سطح اضطراب بر اساس رویداد مهم اخیر در زندگی
</h4>

<p dir="rtl" style="text-align: right;">
نتایج نشان می‌دهد که میانگین سطح اضطراب در افرادی که اخیراً رویداد مهمی در زندگی تجربه نکرده‌اند حدود
<span dir="ltr">3.80</span>
و در افرادی که چنین رویدادی را تجربه کرده‌اند حدود
<span dir="ltr">4.04</span>
است.
</p>

<p dir="rtl" style="text-align: right;">
بنابراین، در این دیتاست میانگین سطح اضطراب در گروه دارای رویداد مهم اخیر حدود
<span dir="ltr">0.24</span>
واحد بیشتر مشاهده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
این اختلاف یک مشاهده توصیفی است و به‌تنهایی برای نتیجه‌گیری درباره رابطه علّی یا معناداری آماری کافی نیست.
برای بررسی معناداری این تفاوت، انجام آزمون آماری مناسب ضروری است.
</p>

<h3 dir="rtl" style="text-align: right;">
5.2 ستون های عددی با اضطراب
</h3>

In [ ]:

numeric_cols = [
    'Age',
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)',
    'Stress Level (1-10)',
    'Heart Rate (bpm)',
    'Breathing Rate (breaths/min)',
    'Sweating Level (1-5)',
    'Therapy Sessions (per month)',
    'Diet Quality (1-10)'
]

fig, axes = plt.subplots(6, 2, figsize=(11, 18))
axes = axes.flatten()

box_colors = ['#A8D5E5', '#F4A261']

for i, col in enumerate(numeric_cols):
    bp = clean_df.boxplot(
        column=col,
        by='Target',
        ax=axes[i],
        patch_artist=True,
        return_type='dict'
    )

    for patch, color in zip(bp[col]['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)

    for median in bp[col]['medians']:
        median.set_color('#1a1a1a')
        median.set_linewidth(1.5)

    axes[i].set_title(f'{col} vs Anxiety Status',
                      fontsize=12,
                      fontweight='bold',
                      pad=12)

    axes[i].set_xlabel('is_Anxious')
    axes[i].set_ylabel(col)
    axes[i].grid(axis='y', alpha=0.3)

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

# =========================
# Layout
# =========================
plt.suptitle(
    'Bivariate Analysis: Numeric Variables vs Anxiety Status',
    fontsize=18,
    fontweight='bold',
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.subplots_adjust(hspace=0.45)
plt.show()

<h2 dir="rtl" style="text-align: right;">
6. آزمون های آماری
</h2>

<h3 dir="rtl" style="text-align: right;">
6.1 آزمون همبستگی: عددی با عددی
</h3>

<h3 dir="rtl" style="text-align: right;">
6.2 آزمون
<span dir="ltr">t-test</span> و
<span dir="ltr">ANOVA</span>:
دسته ای با عددی
</h3>

<h3 dir="rtl" style="text-align: right;">
6.3 آزمون
<span dir="ltr">chi-square</span>:
دسته ای با دسته ای
</h3>

<h4 dir="rtl" style="text-align: right;">
بررسی ارتباط سابقه خانوادگی اضطراب با متغیر هدف
</h4>

<p dir="rtl" style="text-align: right;">
برای بررسی وجود رابطه بین دو متغیر دسته‌ای
<span dir="ltr">Family History of Anxiety</span>
و
<span dir="ltr">Target</span>
از آزمون
<span dir="ltr">Chi-Square Test of Independence</span>
استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
فرض صفر
<span dir="ltr">(H0)</span>
بیان می‌کند که دو متغیر از یکدیگر مستقل هستند و رابطه آماری میان آن‌ها وجود ندارد.
فرض مقابل
<span dir="ltr">(H1)</span>
بیان می‌کند که میان این دو متغیر ارتباط آماری وجود دارد.
</p>

<p dir="rtl" style="text-align: right;">
پیش از اجرای آزمون، جدول توافقی
<span dir="ltr">(Contingency Table)</span>
ساخته می‌شود تا فراوانی مشاهده‌شده برای ترکیب دسته‌های دو متغیر بررسی شود.
</p>

In [ ]:
# ساخت جدول توافقی برای نمایش فراوانی ترکیب‌های
# سابقه خانوادگی اضطراب و متغیر هدف
contingency_table = pd.crosstab(
    clean_df["Family History of Anxiety"],
    clean_df["Target"]
)

# نمایش فراوانی‌های مشاهده‌شده
contingency_table

<h4 dir="rtl" style="text-align: right;">
اجرای آزمون Chi-Square
</h4>

<p dir="rtl" style="text-align: right;">
پس از ساخت جدول توافقی، آزمون
<span dir="ltr">Chi-Square Test of Independence</span>
اجرا می‌شود.
این آزمون فراوانی‌های مشاهده‌شده
<span dir="ltr">(Observed Frequencies)</span>
را با فراوانی‌هایی که در صورت مستقل بودن دو متغیر انتظار می‌رود
<span dir="ltr">(Expected Frequencies)</span>
مقایسه می‌کند.
</p>

<p dir="rtl" style="text-align: right;">
اگر اختلاف بین فراوانی‌های مشاهده‌شده و مورد انتظار به اندازه کافی بزرگ باشد، شواهدی علیه فرض استقلال دو متغیر به دست می‌آید.
</p>

In [ ]:
# اجرای آزمون Chi-Square روی جدول توافقی
chi2_stat, p_value, dof, expected = stats.chi2_contingency(
    contingency_table
)

# نمایش آماره آزمون، مقدار p و درجه آزادی
print("Chi-Square Statistic:", chi2_stat)
print("p-value:", p_value)
print("Degrees of Freedom:", dof)

# تبدیل فراوانی‌های مورد انتظار به DataFrame برای نمایش خواناتر
expected_table = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

# نمایش فراوانی‌های مورد انتظار
expected_table

<h4 dir="rtl" style="text-align: right;">
نتیجه آزمون Chi-Square
</h4>

<p dir="rtl" style="text-align: right;">
نتیجه آزمون
<span dir="ltr">Chi-Square Test of Independence</span>
مقدار آماره آزمون برابر با حدود
<span dir="ltr">87.32</span>
و درجه آزادی برابر با
<span dir="ltr">1</span>
را نشان داد.
</p>

<p dir="rtl" style="text-align: right;">
مقدار
<span dir="ltr">p-value</span>
برابر با حدود
<span dir="ltr">9.24 × 10⁻²¹</span>
است که به‌طور قابل‌توجهی کمتر از سطح معناداری
<span dir="ltr">0.05</span>
است.
بنابراین، فرض صفر مبنی بر استقلال دو متغیر رد می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
در نتیجه، در این دیتاست بین
<span dir="ltr">Family History of Anxiety</span>
و
<span dir="ltr">Target</span>
ارتباط آماری معناداری مشاهده می‌شود.
همچنین نسبت افراد با
<span dir="ltr">Target = 1</span>
در گروه دارای سابقه خانوادگی اضطراب بیشتر از گروه بدون سابقه خانوادگی است.
</p>

<p dir="rtl" style="text-align: right;">
این نتیجه نشان‌دهنده وجود ارتباط آماری میان دو متغیر است و به‌تنهایی وجود رابطه علّی را اثبات نمی‌کند.
</p>

<h3 dir="rtl" style="text-align: right;">
6.4 امتیازی: بازه های اطمینان
</h3>

<h2 dir="rtl" style="text-align: right;">
7.
<span dir="ltr">KPI</span>
و فیچرهای تعاملی
</h2>

<h3 dir="rtl" style="text-align: right;">
7.1 استخراج
<span dir="ltr">KPI</span>
و فیچرهای تعاملی از ستون ها
</h3>

<h3 dir="rtl" style="text-align: right;">
7.2 ویژوال
<span dir="ltr">KPI</span>
و فیچرهای جدید
</h3>

<h2 dir="rtl" style="text-align: right;">
8. ویژوال چندمتغیره: ترکیب ویژگی ها و گروه های پرخطر
</h2>

In [ ]:
plot_df = clean_df[['Stress Level (1-10)', 'Sleep Hours', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Stress Level (1-10)',
    y='Sleep Hours',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},
    s=90,
    alpha=0.5,
    edgecolor='white',
    linewidth=0.7
)

plt.title('Stress Level vs Sleep Hours by Anxiety Status',
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Stress Level (1-10)', fontsize=12)
plt.ylabel('Sleep Hours', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11,
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

sns.jointplot(
    data=clean_df,
    x='Stress Level (1-10)',
    y='Sleep Hours',
    hue='Target',
    kind='kde',
    fill=True,
    alpha=0.6
)

In [ ]:
plot_df = clean_df[['Physical Activity (hrs/week)', 'Stress Level (1-10)', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Stress Level (1-10)',
    y='Physical Activity (hrs/week)',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},
    s=90,
    alpha=0.5,
    edgecolor='white',
    linewidth=0.7
)

plt.title('Stress Level vs Physical Activity by Anxiety Status',
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Stress Level (1-10)', fontsize=12)
plt.ylabel('Physical Activity (hrs/week)', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11,
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

sns.jointplot(
    data=clean_df,
    x='Stress Level (1-10)',
    y='Physical Activity (hrs/week)',
    hue='Target',
    kind='kde',
    fill=True,
    alpha=0.6
)

In [ ]:
plot_df = clean_df[['Heart Rate (bpm)', 'Breathing Rate (breaths/min)', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Heart Rate (bpm)',
    y='Breathing Rate (breaths/min)',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},
    s=90,
    alpha=0.5,
    edgecolor='white',
    linewidth=0.7
)

plt.title('Heart Rate vs Breathing Rate by Anxiety Status',
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Heart Rate (bpm)', fontsize=12)
plt.ylabel('Breathing Rate (breaths/min)', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11,
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

sns.jointplot(
    data=clean_df,
    x='Heart Rate (bpm)',
    y='Breathing Rate (breaths/min)',
    hue='Target',
    kind='kde',
    fill=True,
    alpha=0.6
)

In [ ]:
plot_df = clean_df[['Age', 'Gender', 'Target']]

plt.figure(figsize=(10, 7))

sns.boxplot(
    data=plot_df,
    x='Gender',
    y='Age',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},
    width=0.6,
    linewidth=1.3
)

plt.title('Age Distribution by Gender and Anxiety Status',
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Gender', fontsize=12)
plt.ylabel('Age', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11,
           loc='upper right', frameon=True)

plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()



In [ ]:
plot_df = clean_df[['Caffeine Intake (mg/day)', 'Sleep Hours', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Caffeine Intake (mg/day)',
    y='Sleep Hours',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},
    s=90,
    alpha=0.5,
    edgecolor='none',
)

plt.title('Caffeine Intake vs Sleep Hours by Anxiety Status',
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Caffeine Intake (mg/day)', fontsize=12)
plt.ylabel('Sleep Hours', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11,
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

sns.jointplot(
    data=clean_df,
    x='Caffeine Intake (mg/day)',
    y='Sleep Hours',
    hue='Target',
    kind='kde',
    fill=True,
    alpha=0.6
)
plt.show()

In [ ]:
plot_df = clean_df[['Diet Quality (1-10)', 'Physical Activity (hrs/week)', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Diet Quality (1-10)',
    y='Physical Activity (hrs/week)',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},
    s=90,
    alpha=0.3,
    edgecolor='none'
)

plt.title('Diet Quality vs Physical Activity by Anxiety Status',
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Diet Quality (1-10)', fontsize=12)
plt.ylabel('Physical Activity (hrs/week)', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11,
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()


sns.jointplot(
    data=clean_df,
    x='Diet Quality (1-10)',
    y='Physical Activity (hrs/week)',
    hue='Target',
    kind='kde',
    fill=True,
    alpha=0.6
)
plt.show()

<h2 dir="rtl" style="text-align: right;">
9. امتیازی: ویژوال سه متغیره و بیشتر
</h2>

In [ ]:
fig = go.Figure()

for label_value, name, color in [(0, 'not anxious', BLUE), (1, 'anxious', RED)]:
    subset = clean_df[clean_df[LABEL] == label_value]

    fig.add_trace(go.Scatter(
        x=subset['Sleep Hours'],
        y=subset['Stress Level (1-10)'],
        mode='markers',
        name=name,
        marker={
            'color': color,
            'size': subset['Caffeine Intake (mg/day)'] / 40,
            'opacity': 0.4,
            'line': {'width': 0}
        },
        text=subset['Caffeine Intake (mg/day)'],
        hovertemplate='sleep=%{x}<br>stress=%{y}<br>caffeine=%{text}'
    ))

fig.update_layout(
    template=px.defaults.template,
    title='Anxious people (red) cluster at high stress and low sleep; bigger circles (more caffeine) are more common there',
    xaxis_title='Sleep Hours',
    yaxis_title='Stress Level (1-10)',
    height=550,
    plot_bgcolor=BG
)

fig.show()

In [ ]:
fig = px.scatter(
    clean_df,
    x='Sleep Hours',
    y='Stress Level (1-10)',
    color=LABEL,
    size='Caffeine Intake (mg/day)',
    facet_col='Family History of Anxiety',
    color_discrete_map={
        0: BLUE,
        1: RED
    },
    opacity=0.35,
    size_max=20,
    height=550,
    template=px.defaults.template,
    hover_data=[
        'Anxiety Level (1-10)',
        'Target',
        'Caffeine Intake (mg/day)',
        'Family History of Anxiety'
    ]
)

fig.update_layout(
    title='sleep, stress, caffeine (size), label (color), family history (panel)',
    xaxis_title='Sleep Hours',
    yaxis_title='Stress Level (1-10)'
)

fig.show()

In [ ]:
# میانگین اضطراب در هر ترکیب استرس و خواب
sleep_bins = [0, 4, 5, 6, 7, 8, 9, 12]

plot_df = clean_df.copy()
plot_df['Sleep Bin'] = pd.cut(
    plot_df['Sleep Hours'],
    bins=sleep_bins
)

heat_table = plot_df.pivot_table(
    index='Sleep Bin',
    columns='Stress Level (1-10)',
    values=TARGET,
    aggfunc='mean',
    observed=True
)

fig, ax = plt.subplots(figsize=(11, 5))

sns.heatmap(
    heat_table,
    annot=True,
    fmt='.1f',
    cmap='Reds',
    ax=ax,
    linewidths=0.5,
    linecolor='white'
)

ax.set_title(
    'Mean anxiety by stress and sleep: stress 9 to 10 with sleep under 5 hours is the hottest corner',
    fontsize=10,
    color=TEXT
)

plt.tight_layout()
plt.show()